# Import des librairies

In [169]:
import re
from collections import namedtuple, defaultdict

# Lecture du fichier source

In [170]:
with open('real.txt') as fichier:
    file = fichier.read().split('\n')
file

['bn RSHIFT 2 -> bo',
 'lf RSHIFT 1 -> ly',
 'fo RSHIFT 3 -> fq',
 'cj OR cp -> cq',
 'fo OR fz -> ga',
 't OR s -> u',
 'lx -> a',
 'NOT ax -> ay',
 'he RSHIFT 2 -> hf',
 'lf OR lq -> lr',
 'lr AND lt -> lu',
 'dy OR ej -> ek',
 '1 AND cx -> cy',
 'hb LSHIFT 1 -> hv',
 '1 AND bh -> bi',
 'ih AND ij -> ik',
 'c LSHIFT 1 -> t',
 'ea AND eb -> ed',
 'km OR kn -> ko',
 'NOT bw -> bx',
 'ci OR ct -> cu',
 'NOT p -> q',
 'lw OR lv -> lx',
 'NOT lo -> lp',
 'fp OR fv -> fw',
 'o AND q -> r',
 'dh AND dj -> dk',
 'ap LSHIFT 1 -> bj',
 'bk LSHIFT 1 -> ce',
 'NOT ii -> ij',
 'gh OR gi -> gj',
 'kk RSHIFT 1 -> ld',
 'lc LSHIFT 1 -> lw',
 'lb OR la -> lc',
 '1 AND am -> an',
 'gn AND gp -> gq',
 'lf RSHIFT 3 -> lh',
 'e OR f -> g',
 'lg AND lm -> lo',
 'ci RSHIFT 1 -> db',
 'cf LSHIFT 1 -> cz',
 'bn RSHIFT 1 -> cg',
 'et AND fe -> fg',
 'is OR it -> iu',
 'kw AND ky -> kz',
 'ck AND cl -> cn',
 'bj OR bi -> bk',
 'gj RSHIFT 1 -> hc',
 'iu AND jf -> jh',
 'NOT bs -> bt',
 'kk OR kv -> kw',
 'ks AN

# Nettoyage

In [171]:
tuple_instr = namedtuple('tuple_instr',['type', 'calcul'])
dict_instr = {}
for row in file:
    calcul, destination = row.split(' -> ')

    # On stocke le premier element de la chaine
    first_element = calcul.split()[0]

    # NOT
    if first_element.startswith('N'):
        clean_instr = tuple_instr('NOT', calcul.split()[1])

    # Gate (AND, OR...)
    elif any(c.isupper() for c in calcul) :
        left_gate, gate, right_gate = calcul.split()
        clean_instr = tuple_instr(gate, (left_gate, right_gate))

    # Value to wire (ex: 123 -> x)
    elif first_element.isdigit(): 
        clean_instr = tuple_instr('signal', int(calcul))

    # Wire to wire (ex : lx -> a)
    else:
        clean_instr = tuple_instr('w_to_w', calcul)
    
    
    dict_instr[destination] = clean_instr
dict_instr

{'bo': tuple_instr(type='RSHIFT', calcul=('bn', '2')),
 'ly': tuple_instr(type='RSHIFT', calcul=('lf', '1')),
 'fq': tuple_instr(type='RSHIFT', calcul=('fo', '3')),
 'cq': tuple_instr(type='OR', calcul=('cj', 'cp')),
 'ga': tuple_instr(type='OR', calcul=('fo', 'fz')),
 'u': tuple_instr(type='OR', calcul=('t', 's')),
 'a': tuple_instr(type='w_to_w', calcul='lx'),
 'ay': tuple_instr(type='NOT', calcul='ax'),
 'hf': tuple_instr(type='RSHIFT', calcul=('he', '2')),
 'lr': tuple_instr(type='OR', calcul=('lf', 'lq')),
 'lu': tuple_instr(type='AND', calcul=('lr', 'lt')),
 'ek': tuple_instr(type='OR', calcul=('dy', 'ej')),
 'cy': tuple_instr(type='AND', calcul=('1', 'cx')),
 'hv': tuple_instr(type='LSHIFT', calcul=('hb', '1')),
 'bi': tuple_instr(type='AND', calcul=('1', 'bh')),
 'ik': tuple_instr(type='AND', calcul=('ih', 'ij')),
 't': tuple_instr(type='LSHIFT', calcul=('c', '1')),
 'ed': tuple_instr(type='AND', calcul=('ea', 'eb')),
 'ko': tuple_instr(type='OR', calcul=('km', 'kn')),
 'bx': t

# Traitement

In [ ]:
def calculer_valeur(in_letter : int | str,
                    in_dict_fil_values : dict,
                    in_dict_instructions : dict):
    """

    Pour une lettre donnée, cherche la valeur de celle ci :
        - Cas 1 : La valeur est brute (ex: 123)
        - Cas 2 : La valeur est un calcul d'autres lettres (ex : a AND b)
            - Cas 2.a : La valeur a déja été calculée et est stockée dans dict_fil_values
            - Cas 2.b : Il faut trouver la valeur numérique des lettres (a,b)

    Args:
        in_letter (int | str): Lettre dont on cherche la valeur numérique
        dict_fil_values (defaultdict[int]): Contient les (lettres : valueurs) déja trouvées
        in_dict_instructions (dict): Contient les instructions 'brutes' (calcul associé à aux lettres, ex : c = b + a)

    """
    # print()
    # print('in_letter',in_letter)
    # print('dict_fil_values1',in_dict_fil_values)

    # Si la lettre est un entier
    if in_letter.isdigit():
        # print('la lettre est un entier')
        return int(in_letter)

    # Si on a deja trouvé la valeur associée a la lettre, pas besoin de la recalculer
    if in_letter in in_dict_fil_values:
        # print('valeur de la lettre deja calculée')
        return in_dict_fil_values[in_letter]
    
    # On cherche la formule pour calculer la valeur associée à la lettre ---> la formule correspond au namedtuple
    operation_letter = in_dict_instructions[in_letter]
    # print('operation letter',operation_letter)
    
    # Si on tombre sur un entier brut (123 -> a), on arrete la récursivité
    if operation_letter.type == 'signal':
        return int(operation_letter.calcul)
    
    # Si la partie gache n'est pas un entier, c'est une lettre, il faut donc chercher la valeur de cette lettre
    
    # b -> a
    elif operation_letter.type == 'w_to_w':

        in_dict_fil_values[in_letter] = calculer_valeur(in_letter = operation_letter.calcul,
                                                        in_dict_instructions = in_dict_instructions,
                                                        in_dict_fil_values = in_dict_fil_values)

    # NOT b -> a
    elif operation_letter.type == 'NOT':
        in_dict_fil_values[in_letter] = ~ calculer_valeur(in_letter = operation_letter.calcul,
                                                          in_dict_instructions = in_dict_instructions,
                                                          in_dict_fil_values = in_dict_fil_values)  & 0xFFFF


    # Si le calcul se fait avec 2 opérandes    
    else:
        # On récupère les deux composantes de l'opération
        operandes = operation_letter.calcul
        operande_a = operandes[0]
        operande_b = operandes[1]
        # print('operande_a', operande_a)
        # print('operande_b', operande_b)

        # Pour chaque cpmposante, on cherche sa valeur numérique
        operande_a_value = calculer_valeur(in_letter = operande_a,
                                           in_dict_instructions = in_dict_instructions,
                                           in_dict_fil_values = in_dict_fil_values) 
        operande_b_value = calculer_valeur(in_letter = operande_b,
                                           in_dict_instructions = in_dict_instructions,
                                           in_dict_fil_values = in_dict_fil_values)
        
        # print('operande_a_value', operande_a_value)
        # print('operande_b_value', operande_b_value)


        # a AND b -> c
        if operation_letter.type == 'AND':
            # On effectue l'opération sur leurs valeur numérique
            in_dict_fil_values[in_letter] = operande_a_value & operande_b_value    
    
        # a OR b -> c
        elif operation_letter.type == 'OR':
            # On effectue l'opération sur leurs valeur numérique
            in_dict_fil_values[in_letter] = operande_a_value | operande_b_value

    
        elif operation_letter.type == 'LSHIFT':       
            in_dict_fil_values[in_letter] = operande_a_value << operande_b_value

    
        elif operation_letter.type == 'RSHIFT':         
            in_dict_fil_values[in_letter] = operande_a_value >> operande_b_value
    
    return in_dict_fil_values[in_letter]
    



In [173]:
dict_fil_values = {}

for destination, tuple_instr in dict_instr.items():
    dict_fil_values[destination] = calculer_valeur(in_letter = destination,
                                                   in_dict_instructions = dict_instr,
                                                   in_dict_fil_values = dict_fil_values)

tuple_instr(type='w_to_w', calcul='lx')
tuple_instr(type='w_to_w', calcul='lx')


# Traitement

In [174]:
dict_fil_values['a']

46065